In [ ]:
import os
import numpy as np

# set n and k
n = int(1e3)
k = int(1e1)

# set folder name based on n
n_readable = f"{n:.0e}".replace(".0", "").replace("+0", "")
experiment_dim = "dim_" + n_readable
folder = "./private/data/"+experiment_dim+"/"

# check if folder exists
if not os.path.exists(folder):
    raise FileNotFoundError(f"Folder {folder} does not exist. Please run generate_matrices.ipynb and generate_vectors.ipynb first to generate the matrices.")

# check if n/k has no remainder
if n % k == 0:
    # if it has no remainder, we can simply divide n by k
    m = n // k
    sizes = [n // k] * k
else: 
    raise ValueError("n must be divisible by k")


print(f"n: {n}, k: {k}, sizes: {sizes}")

n: 1000, k: 10, sizes: [100, 100, 100, 100, 100, 100, 100, 100, 100, 100]


# Choose well/ill case and scenario

In [ ]:
# Forza Gurobi a usare un metodo specifico:
# -1 = Automatico (Default)
# 0 = Primal Simplex (Active-Set)
# 1 = Dual Simplex (Active-Set)
# 2 = Barrier (Metodo a punto interno)

params = {
    "matrix": "Q_well",
    "vector": "q_well_sc2",
    "method": "1"
}

In [ ]:
# load Q
Q = np.load(folder+"matrices.npz")[params["matrix"]]

In [ ]:
# load q
q = np.load(folder+"vectors.npz")[params["vector"]]

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import scipy.sparse as sp

model = gp.Model("proj_33")
x = model.addMVar(shape=n, vtype=GRB.CONTINUOUS, lb=0.0, name="x")
obiettivo = (x.T @ Q @ x) + (q.T @ x)
model.setObjective(obiettivo, GRB.MINIMIZE)

# Costruiamo il vettore b di dimensione K, tutto di 1
b = np.ones(k)

# Costruiamo la matrice A (K righe, n colonne). 
# np.eye(K) crea la matrice identità, np.repeat ne ripete ogni colonna m volte.
A = np.repeat(np.eye(k), m, axis=1)

# Convertiamo A in matrice sparsa per ottimizzare drasticamente la memoria
A_sparse = sp.csr_matrix(A)

# Aggiungiamo l'intero blocco di vincoli Ax = b in una sola riga matriciale
model.addConstr(A_sparse @ x == b, name="vincoli_simplessi")

# apply method parameter
model.Params.Method = int(params["method"])

# logfile name path
log_path = folder + "logs/" + params["matrix"] + "_" + params["vector"] + "_" + params["method"] + "_gurobi.log"
model.setParam("LogFile", log_path)

Set parameter Method to value 0
Set parameter LogFile to value "./private/data/dim_1e3/Q_well_q_well_sc2_0_gurobi.log"


In [ ]:
# Avvia la risoluzione
model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D125)

CPU model: Apple M1 Max
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
Method  0

Academic license 2788575 - for non-commercial use only - registered to s.___@studenti.unipi.it
Optimize a model with 10 rows, 1000 columns and 1000 nonzeros (Min)
Model fingerprint: 0x97b1e1f5
Model has 1000 linear objective coefficients
Model has 500500 quadratic objective terms
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [9e-02, 2e+02]
  QObjective range [8e-07, 6e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]

Presolve time: 0.04s
Presolved: 10 rows, 1000 columns, 1000 nonzeros
Presolved model has 500500 quadratic objective terms

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s
       0    1.2817666e+02   0.000000e+00   5.513

In [ ]:
# ==========================================
# 7. ESTRAZIONE DEI RISULTATI
# ==========================================
if model.Status == GRB.OPTIMAL:
    valore_ottimo = model.ObjVal
    x_ottimo_trovato = x.X  # Estrae l'array numpy con i valori della soluzione vincolata x*
    
    print(f"Valore ottimo trovato: {valore_ottimo}")
else:
    print("Ottimizzazione non terminata con successo.")

Valore ottimo trovato: -1489.903247120851
